# 02 · Pelatihan U-Net dan DeepLabV3+ (T4-A, Raabin-WBC)

**Cara pakai (tiap anggota, di akun Colab masing-masing)**
1. Ketua sudah menaruh `data.py`, `runlog.py` (dan folder `splits/`) di folder Drive bersama. Taruh `metrics.py`, `models.py`, `train.py` di folder `code/` yang sama.
2. Runtime → Change runtime type → **T4 GPU**.
3. Isi `OWNER` di sel setup dengan **satu nama tetap** (mis. nama depan). Nama ini menentukan file log `logs/log_<OWNER>.csv`.
4. Jalankan sel setup, lalu **sel uji cepat**. Setelah itu jalankan hanya sel milikmu.

**Aturan**
- Satu faktor berubah per perbandingan, sisanya tetap pada baseline (U-Net ResNet-34, CE+Dice, split `grouped`, augmentasi warna aktif, 256 px, seed 42).
- Semua pemilihan epoch memakai **validasi**. `eval_test=True` hanya untuk konfigurasi final, sekali, di akhir.
- Jangan mengubah file di `splits/`.
- Run yang gagal tetap tercatat otomatis. Run yang terputus karena sesi Colab habis TIDAK tercatat, catat manual.

In [1]:
# ===== SEL SETUP (jalankan tiap sesi Colab baru) =====
from google.colab import drive; drive.mount('/content/drive')
!apt-get install -y unrar > /dev/null 2>&1
!pip -q install albumentations segmentation-models-pytorch
DRIVE = "/content/drive/MyDrive/PROYEK UAS DEEP LEARNING"

!mkdir -p /content/raabin_gt/bmp
!unrar x -inul "{DRIVE}/GroundTruths_bmpformat.rar" /content/raabin_gt/bmp/

import sys; sys.path.append(f"{DRIVE}/code")
import warnings; warnings.filterwarnings("ignore")
from train import run_experiment, run_seeds

OWNER = "Iklil"      # satu nama tetap, tanpa spasi
assert OWNER and " " not in OWNER, "OWNER tidak boleh kosong atau mengandung spasi"
print("OWNER =", OWNER)

Mounted at /content/drive
OWNER = Iklil


## Uji cepat (1 epoch)
Memastikan GPU, unduhan bobot pralatih, dan penulisan log berjalan. Tercatat di log dengan `ablation='smoke_test'`; baris ini dibuang saat laporan. Perkiraan waktu 1-3 menit (belum diukur di T4).

In [ ]:
run_experiment(OWNER, "smoke_test", "-", epochs=1)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

ep  1/1 | loss 0.7121 | val dice cyto 0.8160 nuc 0.9425 | skor 0.8792


{'val_dice_bg': 0.9874,
 'val_iou_bg': 0.9755,
 'val_dice_cyto': 0.816,
 'val_dice_cyto_sd': 0.1045,
 'val_iou_cyto': 0.7007,
 'val_dice_nuc': 0.9425,
 'val_dice_nuc_sd': 0.0398,
 'val_iou_nuc': 0.8938,
 'val_miou': 0.8566,
 'val_miou_sd': 0.0501,
 'val_hd95_cyto': 21.089,
 'val_hd95_cyto_sd': 17.7731,
 'val_hd95_nuc': 12.8683,
 'val_hd95_nuc_sd': 15.2605,
 'run_id': 'Iklil_1123_070529',
 'best_epoch': 1}

## Baseline (dijalankan SEKALI untuk seluruh tim)
Baseline U-Net milik **Anggota 2**, baseline DeepLabV3+ milik **Anggota 3**. Semua ablasi dibandingkan dengan baseline U-Net ini. Jangan dijalankan ulang oleh anggota lain.

In [ ]:
# Baseline U-Net (Anggota 2)
run_experiment(OWNER, "baseline", "unet_ce_dice", arch="unet")

In [ ]:
# Baseline DeepLabV3+ (Anggota 3)
run_experiment(OWNER, "baseline", "deeplabv3plus_ce_dice", arch="deeplabv3plus")

ep  1/20 | loss 0.4728 | val dice cyto 0.7915 nuc 0.9650 | skor 0.8782
ep  2/20 | loss 0.1837 | val dice cyto 0.8099 nuc 0.9662 | skor 0.8881
ep  3/20 | loss 0.1545 | val dice cyto 0.8430 nuc 0.9566 | skor 0.8998
ep  4/20 | loss 0.1476 | val dice cyto 0.8257 nuc 0.9621 | skor 0.8939
ep  5/20 | loss 0.1336 | val dice cyto 0.8432 nuc 0.9595 | skor 0.9013
ep  6/20 | loss 0.1268 | val dice cyto 0.8538 nuc 0.9668 | skor 0.9103
ep  7/20 | loss 0.1291 | val dice cyto 0.8529 nuc 0.9566 | skor 0.9047
ep  8/20 | loss 0.1178 | val dice cyto 0.8734 nuc 0.9683 | skor 0.9208
ep  9/20 | loss 0.1128 | val dice cyto 0.8791 nuc 0.9706 | skor 0.9248
ep 10/20 | loss 0.1094 | val dice cyto 0.8789 nuc 0.9702 | skor 0.9245
ep 11/20 | loss 0.1078 | val dice cyto 0.8588 nuc 0.9615 | skor 0.9102
ep 12/20 | loss 0.1120 | val dice cyto 0.8857 nuc 0.9676 | skor 0.9266
ep 13/20 | loss 0.1066 | val dice cyto 0.8924 nuc 0.9715 | skor 0.9320
ep 14/20 | loss 0.1018 | val dice cyto 0.8849 nuc 0.9708 | skor 0.9279
ep 15/

{'val_dice_bg': 0.9942,
 'val_iou_bg': 0.9885,
 'val_dice_cyto': 0.8933,
 'val_dice_cyto_sd': 0.0821,
 'val_iou_cyto': 0.8155,
 'val_dice_nuc': 0.972,
 'val_dice_nuc_sd': 0.0108,
 'val_iou_nuc': 0.9458,
 'val_miou': 0.9166,
 'val_miou_sd': 0.0385,
 'val_hd95_cyto': 9.4114,
 'val_hd95_cyto_sd': 16.0064,
 'val_hd95_nuc': 3.9822,
 'val_hd95_nuc_sd': 1.178,
 'run_id': 'Iklil_1123_070604',
 'best_epoch': 18}

## Ablasi per anggota (U-Net, 1 seed = 42, evaluasi di VALIDASI)
Nilai yang sama dengan baseline tidak dijalankan lagi. Jika memori GPU habis (terutama 384 px), turunkan `batch_size` ke 8 dan itu tercatat di log.

In [ ]:
# Anggota 1 (Ketua): skema split. Baseline memakai 'grouped'; ini menambah 'random' (bocor antar-apusan)
run_experiment(OWNER, "split", "random", split="random")

In [ ]:
# Anggota 2: fungsi loss. Baseline = ce_dice; tambahkan tiga lainnya
for l in ["ce", "dice", "focal_dice"]:
    run_experiment(OWNER, "loss", l, loss=l)

In [ ]:
# Anggota 3: augmentasi warna. Baseline = aktif; ini menambah 'nonaktif'
run_experiment(OWNER, "color_aug", "off", color_aug=False)

ep  1/20 | loss 0.6837 | val dice cyto 0.8010 nuc 0.9643 | skor 0.8827
ep  2/20 | loss 0.1915 | val dice cyto 0.7859 nuc 0.9717 | skor 0.8788
ep  3/20 | loss 0.1569 | val dice cyto 0.8613 nuc 0.9703 | skor 0.9158
ep  4/20 | loss 0.1374 | val dice cyto 0.8549 nuc 0.9672 | skor 0.9110
ep  5/20 | loss 0.1395 | val dice cyto 0.8685 nuc 0.9558 | skor 0.9122
ep  6/20 | loss 0.1212 | val dice cyto 0.8770 nuc 0.9704 | skor 0.9237
ep  7/20 | loss 0.1157 | val dice cyto 0.8531 nuc 0.9695 | skor 0.9113
ep  8/20 | loss 0.1116 | val dice cyto 0.8757 nuc 0.9660 | skor 0.9208
ep  9/20 | loss 0.1083 | val dice cyto 0.8429 nuc 0.9691 | skor 0.9060
ep 10/20 | loss 0.1125 | val dice cyto 0.8776 nuc 0.9697 | skor 0.9237
ep 11/20 | loss 0.1044 | val dice cyto 0.8742 nuc 0.9636 | skor 0.9189
early stopping di epoch 11 (terbaik: epoch 6)


{'val_dice_bg': 0.9927,
 'val_iou_bg': 0.9856,
 'val_dice_cyto': 0.877,
 'val_dice_cyto_sd': 0.1023,
 'val_iou_cyto': 0.7927,
 'val_dice_nuc': 0.9704,
 'val_dice_nuc_sd': 0.0143,
 'val_iou_nuc': 0.9429,
 'val_miou': 0.9071,
 'val_miou_sd': 0.0457,
 'val_hd95_cyto': 11.8738,
 'val_hd95_cyto_sd': 17.8113,
 'val_hd95_nuc': 4.2595,
 'val_hd95_nuc_sd': 1.3091,
 'run_id': 'Iklil_1123_071117',
 'best_epoch': 6}

In [ ]:
# Anggota 3
run_experiment(OWNER, "color_aug", "off_p20", color_aug=False, patience=20)

ep  1/20 | loss 0.6663 | val dice cyto 0.8133 nuc 0.9550 | skor 0.8841
ep  2/20 | loss 0.1926 | val dice cyto 0.8242 nuc 0.9705 | skor 0.8974
ep  3/20 | loss 0.1552 | val dice cyto 0.8565 nuc 0.9670 | skor 0.9118
ep  4/20 | loss 0.1432 | val dice cyto 0.8704 nuc 0.9717 | skor 0.9210
ep  5/20 | loss 0.1347 | val dice cyto 0.8366 nuc 0.9638 | skor 0.9002
ep  6/20 | loss 0.1220 | val dice cyto 0.8750 nuc 0.9687 | skor 0.9219
ep  7/20 | loss 0.1192 | val dice cyto 0.8559 nuc 0.9700 | skor 0.9129
ep  8/20 | loss 0.1113 | val dice cyto 0.8662 nuc 0.9624 | skor 0.9143
ep  9/20 | loss 0.1066 | val dice cyto 0.8782 nuc 0.9686 | skor 0.9234
ep 10/20 | loss 0.1079 | val dice cyto 0.8841 nuc 0.9683 | skor 0.9262
ep 11/20 | loss 0.1029 | val dice cyto 0.8837 nuc 0.9690 | skor 0.9264
ep 12/20 | loss 0.1005 | val dice cyto 0.8855 nuc 0.9683 | skor 0.9269
ep 13/20 | loss 0.0958 | val dice cyto 0.8838 nuc 0.9725 | skor 0.9282
ep 14/20 | loss 0.0944 | val dice cyto 0.8862 nuc 0.9710 | skor 0.9286
ep 15/

{'val_dice_bg': 0.9943,
 'val_iou_bg': 0.9888,
 'val_dice_cyto': 0.8963,
 'val_dice_cyto_sd': 0.0853,
 'val_iou_cyto': 0.8212,
 'val_dice_nuc': 0.9738,
 'val_dice_nuc_sd': 0.0107,
 'val_iou_nuc': 0.9491,
 'val_miou': 0.9197,
 'val_miou_sd': 0.0403,
 'val_hd95_cyto': 9.1328,
 'val_hd95_cyto_sd': 15.6887,
 'val_hd95_nuc': 4.0696,
 'val_hd95_nuc_sd': 1.6607,
 'run_id': 'Iklil_1123_073544',
 'best_epoch': 17}

In [ ]:
# Anggota 4: resolusi masukan. Baseline = 256; ini menambah 384
run_experiment(OWNER, "resolution", "384", img_size=384)

In [ ]:
# Opsional bila waktu cukup: encoder pralatih vs inisialisasi acak
run_experiment(OWNER, "pretrained", "random_init", pretrained=False)

## Perbandingan utama: 3 seed untuk dua konfigurasi terbaik
Pilih dua konfigurasi berdasarkan hasil **validasi** (misalnya U-Net vs DeepLabV3+ dengan pengaturan terbaik). `eval_test=True` menyentuh set uji, jadi jalankan ini **hanya setelah semua ablasi selesai dan konfigurasinya final**, sekali saja. `save_path` menyimpan bobot untuk model card (satu file ~100 MB).

In [ ]:
# Ganti argumen sesuai konfigurasi final tim (contoh di bawah = baseline)
df_unet, tab_unet = run_seeds(owner=OWNER, ablation="main", factor_value="unet_final",
                              arch="unet", eval_test=True,
                              save_path=f"{DRIVE}/checkpoints/unet_final.pt")

In [ ]:
# anggota 3
import pandas as pd
res = []
for s in (42, 43, 44):
    res.append(run_experiment(OWNER, "main", "deeplabv3plus_final", arch="deeplabv3plus", seed=s,
                              eval_test=True,
                              save_path=f"{DRIVE}/checkpoints/deeplabv3plus_final_seed{s}.pt"))
df_dlv3 = pd.DataFrame(res)
cols = [c for c in df_dlv3.columns if c.startswith(("val_", "test_")) and not c.endswith("_sd")]
print(pd.DataFrame({"mean": df_dlv3[cols].mean(), "sd": df_dlv3[cols].std(ddof=1)}).round(4))

ep  1/20 | loss 0.4770 | val dice cyto 0.8011 nuc 0.9668 | skor 0.8840
ep  2/20 | loss 0.1780 | val dice cyto 0.7345 nuc 0.9637 | skor 0.8491
ep  3/20 | loss 0.1592 | val dice cyto 0.8107 nuc 0.9575 | skor 0.8841
ep  4/20 | loss 0.1450 | val dice cyto 0.8722 nuc 0.9686 | skor 0.9204
ep  5/20 | loss 0.1371 | val dice cyto 0.8675 nuc 0.9680 | skor 0.9178
ep  6/20 | loss 0.1277 | val dice cyto 0.8712 nuc 0.9567 | skor 0.9140
ep  7/20 | loss 0.1226 | val dice cyto 0.8520 nuc 0.9590 | skor 0.9055
ep  8/20 | loss 0.1267 | val dice cyto 0.8729 nuc 0.9698 | skor 0.9214
ep  9/20 | loss 0.1188 | val dice cyto 0.8732 nuc 0.9688 | skor 0.9210
ep 10/20 | loss 0.1115 | val dice cyto 0.8812 nuc 0.9698 | skor 0.9255
ep 11/20 | loss 0.1103 | val dice cyto 0.8677 nuc 0.9697 | skor 0.9187
ep 12/20 | loss 0.1067 | val dice cyto 0.8804 nuc 0.9686 | skor 0.9245
ep 13/20 | loss 0.1032 | val dice cyto 0.8861 nuc 0.9698 | skor 0.9280
ep 14/20 | loss 0.1009 | val dice cyto 0.8849 nuc 0.9695 | skor 0.9272
ep 15/

In [2]:
# anggota 3
for s in (42, 43, 44):
    run_experiment(OWNER, "cross_session", "deeplabv3plus_cs_p20", arch="deeplabv3plus", seed=s,
                   split="cross_session", eval_test=True, patience=20)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

ep  1/20 | loss 0.5685 | val dice cyto 0.7502 nuc 0.9222 | skor 0.8362
ep  2/20 | loss 0.2008 | val dice cyto 0.8048 nuc 0.9577 | skor 0.8812
ep  3/20 | loss 0.1701 | val dice cyto 0.7865 nuc 0.9626 | skor 0.8745
ep  4/20 | loss 0.1472 | val dice cyto 0.8515 nuc 0.9622 | skor 0.9069
ep  5/20 | loss 0.1472 | val dice cyto 0.8511 nuc 0.9666 | skor 0.9088
ep  6/20 | loss 0.1343 | val dice cyto 0.8611 nuc 0.9583 | skor 0.9097
ep  7/20 | loss 0.1231 | val dice cyto 0.8861 nuc 0.9682 | skor 0.9271
ep  8/20 | loss 0.1250 | val dice cyto 0.8775 nuc 0.9635 | skor 0.9205
ep  9/20 | loss 0.1157 | val dice cyto 0.8864 nuc 0.9664 | skor 0.9264
ep 10/20 | loss 0.1147 | val dice cyto 0.8840 nuc 0.9653 | skor 0.9246
ep 11/20 | loss 0.1104 | val dice cyto 0.8931 nuc 0.9683 | skor 0.9307
ep 12/20 | loss 0.1058 | val dice cyto 0.8901 nuc 0.9656 | skor 0.9279
ep 13/20 | loss 0.1036 | val dice cyto 0.8916 nuc 0.9651 | skor 0.9284
ep 14/20 | loss 0.1014 | val dice cyto 0.8984 nuc 0.9686 | skor 0.9335
ep 15/

## Gabung seluruh log (Ketua, di akhir)
Menghasilkan tabel log eksperimen, total jam GPU, dan bukti bahwa set uji hanya dipakai pada run final.

In [ ]:
import glob, pandas as pd
log = pd.concat([pd.read_csv(f) for f in glob.glob(f"{DRIVE}/logs/log_*.csv")], ignore_index=True)
log = log[log.ablation != "smoke_test"]
log.to_csv(f"{DRIVE}/logs/LOG_GABUNGAN.csv", index=False)
print("Total run:", len(log), "| gagal:", (log.status == "failed").sum())
print("Total jam GPU:", round(log.gpu_minutes.sum() / 60, 2))
print("Run yang memakai set uji:", int(log.used_test.sum()))
print(log.groupby("owner").gpu_minutes.sum().div(60).round(2))